In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold

In [ ]:
# Load CSV with two header rows
df = pd.read_excel("../Datasets/excel/GPS_Dataset_3D_8_Channels_Authentic_and_Simulated.xlsx", header=[0, 1])

# List of variables/features to plot (excluding 'Output')
features = [
    "PRN", "Carrier_Doppler_hz", "Pseudorange_m", "RX_time", "TOW_at_current_symbol_s",
    "Carrier_phase_cycles", "EC", "LC", "PC", "PIP", "PQP", "TCD", "CN0"
]

In [ ]:
print(df.columns.tolist())

In [ ]:
for features in features:
    # Find all channel columns for this variable
    channels = [col[1] for col in df.columns if col[0] == features]
    if not channels:
        continue  # Skip if none found

    plt.figure(figsize=(12, 6))
    for ch in channels:
        series = df[(features, ch)]
        if pd.api.types.is_numeric_dtype(series):
            plt.plot(series, label=f"ch{ch}")
    plt.xlabel("Sample")
    plt.ylabel(features)
    plt.title(f"{features} for all channels")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
output_channels = [col[1] for col in df.columns if col[0] == "Output"]

for ch in output_channels:
    output_series = df[('Output', ch)]

    # Clean and extract class labels
    output_series = output_series.astype(str).str.strip()
    output_series = output_series.str.extract(r'(\d+)')[0]
    output_series = pd.to_numeric(output_series, errors='coerce')

    vc = output_series.value_counts().sort_index()
    vc.index = vc.index.astype(int)

    plt.figure(figsize=(6, 4))
    plt.bar(vc.index, vc.values)
    plt.xlabel("Class Label")
    plt.ylabel("Count")
    plt.title(f"Class Distribution in Output ({ch})")
    plt.xticks(vc.index)
    plt.tight_layout()
    plt.show()

# Class meanings:
# 0 = authentic GPS signal
# 1 = simple spoofing attack
# 2 = intermediate spoofing attack
# 3 = advanced spoofing attack

In [ ]:
for ch in output_channels:
    series = df[('Output', ch)].astype(str).str.strip().str.extract(r'(\d+)')[0]
    series = pd.to_numeric(series, errors='coerce')
    class_labels = sorted(series.dropna().unique())
    class_labels = [int(x) for x in class_labels]
    print(f"Channel {ch}: Class labels: {class_labels}")

In [ ]:
# Label column
y = df[('Output', 'ch0')].astype(str).str.strip().str.extract(r'(\d+)')[0]
y = pd.to_numeric(y, errors='coerce')

# Drop only the Output ch0 column
X = df.drop(('Output', 'ch0'), axis=1)

# K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    print(f"\nFold {fold}:")
    print("Train class distribution:")
    print(y_train.value_counts(normalize=True).sort_index())
    print("Test class distribution:")
    print(y_test.value_counts(normalize=True).sort_index())